# Boundary Conditions and Convergence

Use boundary data and resolution changes to test whether wave-equation errors decrease as expected.

Navigation: [Index](../index.ipynb) | Previous: [Method of Lines and Runge-Kutta](method_of_lines_and_rk.ipynb) | Next: [Finite-Difference Playground](finite_difference_playground.ipynb)

## Why This Matters
Stencil points near an edge need boundary data. A convergence test checks that the numerical error decreases when the grid is refined.

## What You Will See
You will see the real Cartesian generator, two short runs at different resolutions, and a complete diagnostic comparison table.

Prerequisite bridge: this notebook follows [Method of Lines and Runge-Kutta](method_of_lines_and_rk.ipynb). Terms are defined here before they are used.

## Generate, Run, and Compare Diagnostics
The Cartesian wave project uses periodic boundary handling in its compact test case. Running with convergence factor 2 refines the grid and should reduce the diagnostic error.

In [1]:
from pathlib import Path
import re
import subprocess
import sys
import tempfile

PROJECT_NAME = "wave_equation_cartesian"
workspace_manager = tempfile.TemporaryDirectory(prefix="nrpy_tutorial_cartesian_", dir=Path.cwd())
WORKSPACE = Path(workspace_manager.name)
PROJECT_DIR = WORKSPACE / "project" / PROJECT_NAME

command = [sys.executable, "-m", "nrpy.examples.wave_equation_cartesian"]
print("generator command: python -m nrpy.examples.wave_equation_cartesian")
completed = subprocess.run(
    command,
    cwd=WORKSPACE,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    check=True,
    timeout=300,
)
print(completed.stdout.replace(str(WORKSPACE), "<workspace>"))
assert PROJECT_DIR.is_dir()
print("project path: project/wave_equation_cartesian")

generator command: python -m nrpy.examples.wave_equation_cartesian


Finished! Now go into project/wave_equation_cartesian and type `make` to build, then ./wave_equation_cartesian to run.
    Parameter file can be found in wave_equation_cartesian.par

project path: project/wave_equation_cartesian


In [2]:
parfile = PROJECT_DIR / "wave_equation_cartesian.par"
par_text = parfile.read_text(encoding="utf-8")
par_text = par_text.replace("t_final = 8.0", "t_final = 0.3125")
par_text = par_text.replace("diagnostics_output_every = 0.2", "diagnostics_output_every = 0.15625")
par_text = par_text.replace("output_progress_every = 1", "output_progress_every = 1000000")
parfile.write_text(par_text, encoding="utf-8")
print("--- runtime wave_equation_cartesian.par ---")
print(parfile.read_text(encoding="utf-8", errors="replace"))

subprocess.run(["make", "-j2"], cwd=PROJECT_DIR, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=True, timeout=300)
for args in ([], ["2.0"]):
    subprocess.run([f"./{PROJECT_NAME}", *args], cwd=PROJECT_DIR, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=True, timeout=90)

diagnostic_rows = {}
for diagnostic in sorted(PROJECT_DIR.glob("out0d-conv_factor*.txt")):
    rows = []
    print(f"\n--- {diagnostic.name} ---")
    text = diagnostic.read_text(encoding="utf-8", errors="replace")
    print(text)
    for line in text.strip().splitlines():
        values = [float(value) for value in line.split()]
        rows.append(values)
    diagnostic_rows[diagnostic.name] = rows

coarse_name = "out0d-conv_factor1.00.txt"
fine_name = "out0d-conv_factor2.00.txt"
coarse_times = {round(row[0], 12) for row in diagnostic_rows[coarse_name] if row[0] > 0.0}
fine_times = {round(row[0], 12) for row in diagnostic_rows[fine_name] if row[0] > 0.0}
comparison_time = max(coarse_times.intersection(fine_times))
coarse_row = next(row for row in diagnostic_rows[coarse_name] if round(row[0], 12) == comparison_time)
fine_row = next(row for row in diagnostic_rows[fine_name] if round(row[0], 12) == comparison_time)
coarse_error = coarse_row[1]
fine_error = fine_row[1]
print("comparison time:", f"{comparison_time:.6e}")
print("uu error coarse:", f"{coarse_error:.6e}")
print("uu error fine:", f"{fine_error:.6e}")
print("uu error ratio coarse/fine:", coarse_error / fine_error)
assert fine_error < coarse_error

--- runtime wave_equation_cartesian.par ---
#### wave_equation_cartesian BH@H parameter file. NOTE: only commondata CodeParameters appear here ###
###########################
###########################
### Module: __main__
convergence_factor = 1.0        # (REAL)
diagnostics_output_every = 0.15625  # (REAL)
###########################
###########################
### Module: nrpy.equations.wave_equation.WaveEquation_Solutions_InitialData
sigma = 3.0                     # (REAL)
wavespeed = 1.0                 # (REAL)
###########################
###########################
### Module: nrpy.infrastructures.BHaH.MoLtimestepping.register_all
CFL_FACTOR = 0.5                # (REAL)
t_final = 0.3125                   # (REAL)
###########################
###########################
### Module: nrpy.infrastructures.BHaH.diagnostics.progress_indicator
output_progress_every = 1000000       # (int)




--- out0d-conv_factor1.00.txt ---
0.000000e+00 0.000000e+00 0.000000e+00 3.991879e+00 3.991879e+00
1.562500e-01 4.061498e-08 2.140944e-05 3.983805e+00 3.983805e+00


--- out0d-conv_factor2.00.txt ---
0.000000e+00 0.000000e+00 0.000000e+00 3.997967e+00 3.997967e+00
1.562500e-01 2.797348e-09 1.349218e-06 3.989851e+00 3.989851e+00

comparison time: 1.562500e-01
uu error coarse: 4.061498e-08
uu error fine: 2.797348e-09
uu error ratio coarse/fine: 14.51910166343265


## Interpreting the Output
The comparison uses diagnostic rows at the same physical time. The smaller refined-grid error is the evidence that the boundary-handled finite-difference run is converging.

## Where This Leads
- [Finite-Difference Playground](finite_difference_playground.ipynb)
- [Curvilinear Boundary Conditions](../4-curvilinear/curvilinear_boundary_conditions.ipynb)
- [Start-to-Finish Cartesian Wave Project](../3-wave_equation/start_to_finish_wave_cartesian.ipynb)